# N135 · 树状数组与前缀聚合

**目标：** 利用lowbit分解前缀并实现动态求和。

**先修：** N124, N134, N031。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 一基下标；lowbit；点增量；前缀/区间和；双BIT扩展；逆序计数。

**配套讲解来源：** [同名逐章意见](../../comment/135_fenwick_tree.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决的问题是：我有一个数组，一边不停地"把某个位置加上一个数"，一边随时要问"从开头到某个位置的和是多少"（或者任意一段区间的和）。如果用普通数组，更新是 O(1) 但求和要 O(n)；如果预先存前缀和，求和是 O(1) 但每次更新都要把后面所有前缀和重算一遍。树状数组（Fenwick 树）把两边都做到 O(log n)。

一个具体到数字的例子：数组 `[1, 3, 5]`，查 `range_sum(0, 3)` 得 9（即 1+3+5）；接着执行 `add(1, -1)`（把下标 1 的元素减 1，数组变成 `[1, 2, 5]`），再查 `range_sum(0, 3)` 得 8。更新之后查询立刻反映变化，这就是"动态求和"。n = 10^5 时，一次操作只有约 17 步循环，十万次操作也是一瞬间的事。

本章还把这个结构用在"右侧更小数"问题上：输入 `[5, 2, 6, 1]` → 输出 `[2, 1, 1, 0]`（上一章我们用有序列表做，最坏 O(n²)；这一章用树状数组做到 O(n log n)）。

## 2. 基础知识：先读懂这些词

- **树状数组（Fenwick tree / Binary Indexed Tree）**：一个叫 `bit` 的数组，它不是原数组本身，而是把原数组切成一块一块，每格存"某一块的和"。你要它是因为它能让"加一个数"和"问前缀和"都只用十几次循环。
- **一基下标**：内部数组从 1 开始编号（bit[1..n]），而对外接口仍用零基下标。为什么要一基？因为 lowbit 技巧要求下标的二进制最低位有意义，下标 0 的 lowbit 是 0，会导致循环走不动，所以 0 号位空着不用。
- **lowbit（最低位的 1）**：`i & -i` 利用补码的性质取出 i 的二进制表示里最低那个 1。比如 6 的二进制是 110，`6 & -6 = 2`。它是整个结构的"切刀"：每格 bit[i] 管多大的一块、查询下一步跳到哪，全由它决定。
- **覆盖块**：`bit[i]` 保存原数组半开区间 `[i - lowbit(i), i)` 的和。下标越"整"（二进制尾部零越多），管的块越大。
- **点增量（point update）**：`add(i, delta)` 是"在该位置加上 delta"，不是"把该位置改成 delta"。这是初学最常踩的坑，notebook 特意强调：想把旧值 3 改成 2，应该传 delta = -1。
- **前缀和 / 区间和**：前缀和 `prefix_sum(end)` 是前 end 个元素的和（半开区间 `[0, end)`）；区间和 `range_sum(left, right)` 用"两个前缀相减"得到：`prefix_sum(right) - prefix_sum(left)`。
- **坐标压缩配合计数**：值可能很大或为负，先压缩成小秩（上一章 N134 的内容），秩才能当数组下标用。
- **逆序计数 / 右侧更小数**：从右往左扫，每步问"当前秩之前已经出现了几个数"，这正好是"右边比我小的个数"。

## 3. 思路推导：从小例子开始

- **Step 1**：约定 bit 数组从 1 号开始，`bit[i]` 负责原数组半开区间 `[i - lowbit(i), i)` 这一段的和。这个约定是唯一需要记住的规则，后面所有操作都从它推出来。
- **Step 2（建树）**：先把原值抄进 bit[1..n]，然后从 i = 1 到 n 逐个把 `bit[i]` 的值"上交"给它的父亲 `parent = i + (i & -i)`（如果父亲没越界）。这样每格在轮到它之前，已经收齐了自己管辖块的完整和。
- **Step 3（前缀查询）**：要求前 end 项的和，先拿 `bit[end]`（它盖住最后 lowbit(end) 项），然后 `end -= lowbit(end)` 跳到前一块的格子，重复直到 end 变 0。每一步拿到的块互不重叠，拼起来恰好是整个前缀。
- **Step 4（点更新）**：位置 i 加了 delta 后，所有覆盖 i 的块都得加 delta。从 `i += 1`（换到内部一基下标）开始，做 `bit[i] += delta` 然后 `i += lowbit(i)`，一路向上走遍所有祖先块，直到越界。
- **Step 5（区间和）**：`range_sum(l, r) = prefix_sum(r) - prefix_sum(l)`，减掉前 l 项就剩中间那段。
- **Step 6（右侧更小数）**：先对全部值做坐标压缩得到秩；从右往左扫，对当前元素（秩为 r）先查 `prefix_sum(r)`——它统计的是秩 0..r-1 的出现次数，也就是"严格比我小的数出现了几个"——再执行 `add(r, 1)` 把自己记进去。

**手算演示（“运行示例”部分 的建树过程，a = [1, 3, 5, 7, 9, 11]）**：先把 bit 初始化成 `[0, 1, 3, 5, 7, 9, 11]`（0 号位空着），然后按下标从小到大"上交"：

| i | lowbit(i) | 父亲 i+lowbit(i) | 动作 | 该步之后 bit 的样子 |
|---|---|---|---|---|
| 1 | 1 | 2 | bit[2] += bit[1]（即 +1） | `[0,1,4,5,7,9,11]` |
| 2 | 2 | 4 | bit[4] += bit[2]（即 +4） | `[0,1,4,5,11,9,11]` |
| 3 | 1 | 4 | bit[4] += bit[3]（即 +5） | `[0,1,4,5,16,9,11]` |
| 4 | 4 | 8（越界） | 不动 | 同上 |
| 5 | 1 | 6 | bit[6] += bit[5]（即 +9） | `[0,1,4,5,16,9,20]` |
| 6 | 2 | 8（越界） | 不动 | 最终 `[0,1,4,5,16,9,20]` |

对照 “运行示例”部分 打出的表格：bit[2] 盖住区间 (0,2)，块和 4 = 1+3；bit[4] 盖住 (0,4)，块和 16 = 1+3+5+7；bit[6] 盖住 (4,6)，块和 20 = 9+11；bit[1]、bit[3]、bit[5] 各盖一个格子，就是原值本身。

**再演示一次查询**：`prefix_sum(5)` 怎么算？end = 5，先拿 bit[5] = 9（盖住区间 (4,5)）；end -= lowbit(5) = 5-1 = 4，再拿 bit[4] = 16（盖住 (0,4)）；end -= lowbit(4) = 4-4 = 0，停。9 + 16 = 25，正是 1+3+5+7+9。

**再演示一次更新**（对应 “自动测试”部分 的 `tree.add(1, -1)`，三元素数组 [1,3,5] 建树后 bit = `[0, 1, 4, 5]`）：外部下标 1 换成内部下标 2；bit[2] += -1 → bit[2] = 3；i += lowbit(2) = 2 → i = 4，超过 n=3，循环结束。只改了一格就把"下标 1 减 1"记进了所有相关块（n=3 时 bit[2] 恰好盖住区间 (0,2)，包含下标 1）。更新后再查 range_sum(0,3)：prefix_sum(3) = bit[3] + bit[2] = 5 + 3 = 8，正是 1+2+5。

顺手列一张 small 的 lowbit 速查表，你手算时可以直接对：

| i（二进制） | lowbit(i) = i & -i | 含义 |
|---|---|---|
| 1 (1) | 1 | 只管自己 |
| 2 (10) | 2 | 管 2 格 |
| 3 (11) | 1 | 只管自己 |
| 4 (100) | 4 | 管 4 格 |
| 5 (101) | 1 | 只管自己 |
| 6 (110) | 2 | 管 2 格 |

规律：lowbit 就是二进制最右边那个 1 及其后面的零；下标是 2 的幂时管得最宽（bit[4] 管 4 格），奇数永远只管自己那格。

## 4. 核心代码：Fenwick

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
class Fenwick:

    def __init__(self, values):
        values = [0] * values if isinstance(values, int) else list(values)
        self.n = len(values)
        self.bit = [0] + values
        for i in range(1, self.n + 1):
            parent = i + (i & -i)
            if parent <= self.n:
                self.bit[parent] += self.bit[i]

    def add(self, i, delta):
        if not 0 <= i < self.n:
            raise IndexError('point index outside tree')
        i += 1
        while i <= self.n:
            self.bit[i] += delta
            i += i & -i

    def prefix_sum(self, end):
        if not 0 <= end <= self.n:
            raise IndexError('prefix boundary outside tree')
        answer = 0
        while end:
            answer += self.bit[end]
            end -= end & -end
        return answer

    def range_sum(self, left, right):
        if left > right:
            raise ValueError('invalid range')
        return self.prefix_sum(right) - self.prefix_sum(left)
```

### 逐段读懂代码

“分段实现”部分 有两个部分：Fenwick 类和 count_smaller 函数。

**构造函数 `__init__`**

```python
def __init__(self,values):
        values=[0]*values if isinstance(values,int) else list(values)
        self.n=len(values); self.bit=[0]+values
        for i in range(1,self.n+1):
            parent=i+(i&-i)
            if parent<=self.n: self.bit[parent]+=self.bit[i]
```

第一行是个双用途入口：如果你传进来一个整数（比如 `Fenwick(6)`），它就建一个长度为 6 的全零数组（`[0]*values`）；如果传进来的是列表，就拷贝一份（`list(values)`），避免改动调用方的原数组。`self.bit=[0]+values` 在最前面补一个 0，把一基下标空出来。循环就是第三节手算的"上交"过程：每个 i 把自己的块和加给父亲 `i+(i&-i)`，`if parent<=self.n` 挡住越界的父亲（树根到此为止）。这样建树整体是 O(n)，比做 n 次单点 add 的 O(n log n) 更快。

**点更新 `add`**

```python
def add(self,i,delta):
        if not 0<=i<self.n: raise IndexError('point index outside tree')
        i+=1
        while i<=self.n: self.bit[i]+=delta; i+=i&-i
```

第一行挡非法下标：对外是零基，i 必须在 `[0, n)` 里，负数或太大都直接抛 IndexError。`i+=1` 把外部零基下标换成内部一基下标。循环体一行两句：当前块加上 delta，然后 `i+=i&-i` 跳到下一个覆盖这个位置的更大块，直到跳出树顶。注意语义是"增量"：bit 里没有存原值，想改成具体值得先算差值。

**前缀查询 `prefix_sum`**

```python
def prefix_sum(self,end):
        if not 0<=end<=self.n: raise IndexError('prefix boundary outside tree')
        answer=0
        while end: answer+=self.bit[end]; end-=end&-end
        return answer
```

第一行挡非法边界：end 允许取 0（空前缀）到 n（整个数组）。`while end` 在 end 变 0 时自然停下；每轮 `answer+=self.bit[end]` 拿一块，`end-=end&-end` 跳到上一块。查询方向是"不断清掉最低位的 1"，和 add 的"不断加上 lowbit"方向相反。

**区间查询 `range_sum`**

```python
def range_sum(self,left,right):
        if left>right: raise ValueError('invalid range')
        return self.prefix_sum(right)-self.prefix_sum(left)
```

先挡住左边界大于右边界的非法区间（半开区间 `[left, right)`，left 等于 right 是合法的空区间，结果为 0）。然后两个前缀一减就是答案。

**右侧更小数 `count_smaller`**

```python
def count_smaller(nums):
    ordered = sorted(set(nums))
    ranks = {x: i for i, x in enumerate(ordered)}
    tree = Fenwick(len(ordered))
    answer = []
    for x in reversed(nums):
        r = ranks[x]
        answer.append(tree.prefix_sum(r))
        tree.add(r, 1)
    return answer[::-1]
```

第一行做三件事：坐标压缩（去重排序建值到秩的字典，这是 N134 的套路），建一棵长度为"不同值个数"的全零树（`Fenwick(len(ordered))` 走的是整数入口），准备答案列表。循环从右往左：`r=ranks[x]` 查秩；`tree.prefix_sum(r)` 问"秩 0..r-1 的值出现了几次"，因为不含秩 r 自己，等值元素不会被算进去，所以是"严格更小"；然后 `tree.add(r,1)` 把当前值记一次。最后 `answer[::-1]` 把倒着收集的答案翻正。

## 5. 分段实现：按顺序运行

**本章接口：** `Fenwick.add(i, delta)`、`Fenwick.prefix_sum(end)`、`count_smaller(nums)`

### Fenwick

In [1]:
class Fenwick:

    def __init__(self, values):
        values = [0] * values if isinstance(values, int) else list(values)
        self.n = len(values)
        self.bit = [0] + values
        for i in range(1, self.n + 1):
            parent = i + (i & -i)
            if parent <= self.n:
                self.bit[parent] += self.bit[i]

    def add(self, i, delta):
        if not 0 <= i < self.n:
            raise IndexError('point index outside tree')
        i += 1
        while i <= self.n:
            self.bit[i] += delta
            i += i & -i

    def prefix_sum(self, end):
        if not 0 <= end <= self.n:
            raise IndexError('prefix boundary outside tree')
        answer = 0
        while end:
            answer += self.bit[end]
            end -= end & -end
        return answer

    def range_sum(self, left, right):
        if left > right:
            raise ValueError('invalid range')
        return self.prefix_sum(right) - self.prefix_sum(left)

### count_smaller

In [2]:
def count_smaller(nums):
    ordered = sorted(set(nums))
    ranks = {x: i for i, x in enumerate(ordered)}
    tree = Fenwick(len(ordered))
    answer = []
    for x in reversed(nums):
        r = ranks[x]
        answer.append(tree.prefix_sum(r))
        tree.add(r, 1)
    return answer[::-1]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,3,5,7,9,11]; tree=Fenwick(a)
show_table(['内部下标','覆盖原数组半开区间','块和'],[(i,(i-(i&-i),i),tree.bit[i]) for i in range(1,len(a)+1)])

内部下标,覆盖原数组半开区间,块和
1,"(0, 1)",1
2,"(0, 2)",4
3,"(2, 3)",5
4,"(0, 4)",16
5,"(4, 5)",9
6,"(4, 6)",20


## 7. 正确性、适用条件与复杂度

bit[i]的覆盖块是二进制下标分解决定的。查询沿i−lowbit(i)得到互不重叠且恰覆盖目标前缀的块；点增量更新所有包含该点的祖先块，维持块和不变量。较小数查询用prefix_sum(rank)，不包含等值秩。

**代价：** 线性建树O(n)；更新和前缀/区间查询O(log n)；计数含压缩O(n log n)，空间O(n)。add是增量而非赋值；修改旧值3到新值2应传−1。

### 展开理解

为什么查询是对的：每次 `end -= lowbit(end)` 之后，新 end 恰好是"刚拿走的那块左边一格"，两块拼起来严丝合缝，中间不会漏也不会重叠；循环走到的下标二进制里 1 的个数每次少一个，最终一定到 0。所以拿到的块合起来正好是前 end 项，一项不多一项不少。

为什么更新是对的：`i += lowbit(i)` 走到的下标，其覆盖区间都包含位置 i（可以对着第三节的表验证：改下标 5 的值，bit[5] 和 bit[6] 都要变，而 5+1=6 正是它的父亲）；反过来说，覆盖 i 的块只有这些祖先，一个不漏。所以每次 add 之后，"每格等于自己那块的和"这个约定始终成立。

count_smaller 为什么是对的：处理到位置 i 时，树里记录的恰好是位置 i 右边所有已扫过的元素的出现次数；`prefix_sum(r)` 数的是秩严格小于 r 的出现次数，等于"右边比 x 小的元素个数"，且等值不算（因为区间是半开的，秩 r 自己没被包含）。

复杂度：建树 O(n)。add 和 prefix_sum 每一步都让下标的二进制位数严格变化，循环次数最多约 log2(n)，所以都是 O(log n)。n = 10^5 时 log2(n) ≈ 17，一次操作约 17 步，十万次操作也就两百万步左右，瞬间完成。count_smaller 里每个元素做一次压缩查字典、一次查询、一次更新，整体 O(n log n)，空间 O(n)——对比上一章有序列表版本最坏 O(n²)，n = 10^5 时从一百亿次级别降到两百万次级别。

## 8. 单元测试：每个用例在检查什么

- `tree=Fenwick([1,3,5]); assert tree.range_sum(0,3)==9`：正常值测试，验证建树正确、全区间和等于 1+3+5。
- `tree.add(1,-1); assert tree.range_sum(0,3)==8`：点增量测试，改一个点后区间和立刻反映 -1 的变化，验证更新路径走全了。
- `assert count_smaller([5,2,6,1])==[2,1,1,0]`：压缩 + 树状数组计数的标准用例，和 N134 的有序列表版本结果一致。
- 中间的随机对拍（`rng=Random(135)` 固定随机种子保证可复现）：n 从 1 到 24，每个长度生成值域 [-5, 5] 的随机数组（含负数和重复值），然后做 80 次随机操作——一半概率随机选个点加随机增量（同步更新暴力数组 a），一半概率随机查一段区间并断言和 `sum(a[l:r])` 相等。它同时覆盖了"先更新后查询""连续更新""空区间（l==r）"等交错情形，比手工用例狠得多。每个长度最后还断言 `count_smaller` 和暴力版逐位一致。
- `assert Fenwick(0).prefix_sum(0)==0`：边界测试，空树（走整数入口传 0）查空前缀必须是 0，验证循环和边界检查在 n=0 时不崩。

In [4]:
tree = Fenwick([1, 3, 5])

assert tree.range_sum(0, 3) == 9

tree.add(1, -1)

assert tree.range_sum(0, 3) == 8

assert count_smaller([5, 2, 6, 1]) == [2, 1, 1, 0]

from random import Random

rng = Random(135)

for n in range(1, 25):
    a = [rng.randrange(-5, 6) for _ in range(n)]
    tree = Fenwick(a)
    for _ in range(80):
        if rng.randrange(2):
            i = rng.randrange(n)
            delta = rng.randrange(-5, 6)
            a[i] += delta
            tree.add(i, delta)
        else:
            l = rng.randrange(n + 1)
            r = rng.randrange(l, n + 1)
            assert tree.range_sum(l, r) == sum(a[l:r])
    assert count_smaller(a) == [sum((y < x for y in a[i + 1:])) for i, x in enumerate(a)]

assert Fenwick(0).prefix_sum(0) == 0

print('N135: 所有本章断言通过')

N135: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 实现范围加与点查。

**练习 2：** 更新赋值时先计算增量。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1：实现范围加与点查。** 提示：现在的 add 是"一个点加 delta"，你要的是"一段区间 `[l, r)` 全体加 d，然后查单点的值"。想想差分数组：如果只维护"相邻差"，区间加 d 只需在 l 处 +d、在 r 处 -d 两个动作，而"单点当前值"恰好等于差分数组的前缀和——这个前缀和正是 prefix_sum 已经会算的东西。所以你不需要新结构，只需要换一种"往树里放什么"的语义。手算示例建议用 `[0,0,0,0]` 执行 `[0,2)+3` 再逐点查询。

**练习 2：更新赋值时先计算增量。** 提示：add 传的是增量而不是新值。如果你在外面另存一份原值数组，想"把位置 i 从旧值 old 改成新值 new"，正确的调用是 `add(i, new - old)`。构造用例：建树 `[1,3,5]`，把下标 1 的 3 改成 2，等价于 `add(1, -1)`，区间和应从 9 变 8——先手算，再验证。想清楚为什么直接 `add(1, 2)` 是错的（它会把 3 当成 5）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
class Fenwick:

    def __init__(self, values):
        values = [0] * values if isinstance(values, int) else list(values)
        self.n = len(values)
        self.bit = [0] + values
        for i in range(1, self.n + 1):
            parent = i + (i & -i)
            if parent <= self.n:
                self.bit[parent] += self.bit[i]

    def add(self, i, delta):
        if not 0 <= i < self.n:
            raise IndexError('point index outside tree')
        i += 1
        while i <= self.n:
            self.bit[i] += delta
            i += i & -i

    def prefix_sum(self, end):
        if not 0 <= end <= self.n:
            raise IndexError('prefix boundary outside tree')
        answer = 0
        while end:
            answer += self.bit[end]
            end -= end & -end
        return answer

    def range_sum(self, left, right):
        if left > right:
            raise ValueError('invalid range')
        return self.prefix_sum(right) - self.prefix_sum(left)

def count_smaller(nums):
    ordered = sorted(set(nums))
    ranks = {x: i for i, x in enumerate(ordered)}
    tree = Fenwick(len(ordered))
    answer = []
    for x in reversed(nums):
        r = ranks[x]
        answer.append(tree.prefix_sum(r))
        tree.add(r, 1)
    return answer[::-1]

# 示例与回归测试
tree = Fenwick([1, 3, 5])

assert tree.range_sum(0, 3) == 9

tree.add(1, -1)

assert tree.range_sum(0, 3) == 8

assert count_smaller([5, 2, 6, 1]) == [2, 1, 1, 0]

from random import Random

rng = Random(135)

for n in range(1, 25):
    a = [rng.randrange(-5, 6) for _ in range(n)]
    tree = Fenwick(a)
    for _ in range(80):
        if rng.randrange(2):
            i = rng.randrange(n)
            delta = rng.randrange(-5, 6)
            a[i] += delta
            tree.add(i, delta)
        else:
            l = rng.randrange(n + 1)
            r = rng.randrange(l, n + 1)
            assert tree.range_sum(l, r) == sum(a[l:r])
    assert count_smaller(a) == [sum((y < x for y in a[i + 1:])) for i, x in enumerate(a)]

assert Fenwick(0).prefix_sum(0) == 0

print('N135: 所有本章断言通过')

N135: 所有本章断言通过


## 11. 代表题与迁移

- **307. Range Sum Query - Mutable（区域和检索 - 数组可修改）**：本章 canonical 题，练的就是"点更新 + 区间和"这对操作，`add` 和 `range_sum` 直接对应题目的 `update` 和 `sumRange`。
- **315. Count of Smaller Numbers After Self（右侧更小数）**：本章 canonical 题，练的是"坐标压缩 + 从右往左用前缀计数"这条流水线，即 `count_smaller` 的原题。
- **493. Reverse Pairs（翻转对）**：extension 题，仍然是从右往左（或分治）统计"右边满足某种大小关系的元素个数"，只是比较条件从"严格小"变成"小于 x 的一半"，练的是把前缀计数迁移到新比较关系上。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。